In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash")

In [3]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?")])

AIMessage(content="저는 실시간 주가 정보를 제공할 수 없습니다. 제 지식은 특정 시점까지의 데이터에 기반하고 있으며, 주가는 실시간으로 변동하기 때문입니다.\n\n테슬라(TSLA) 주가가 한 달 전에 비해 올랐는지 내렸는지 확인하시려면, 다음과 같은 금융 정보 웹사이트나 앱을 이용하시면 됩니다:\n\n*   **구글 금융 (Google Finance)**\n*   **네이버 금융 (Naver Finance)**\n*   **카카오 증권 (Kakao Stock)**\n*   **야후 파이낸스 (Yahoo Finance)**\n*   **사용하시는 증권사 앱**\n\n이러한 플랫폼에서 'TSLA' 또는 '테슬라'를 검색하신 후, **1개월(1M)** 단위의 주가 차트나 변동률을 확인하시면 됩니다.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c1317-2299-7af1-964e-b0bc78599aa4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 572, 'total_tokens': 589, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 394}})

In [4]:
from datetime import datetime
from zoneinfo import ZoneInfo

In [6]:
from langchain_core.tools import tool

@tool
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: Asia/Seoul) 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 LLM 답변 생성에 사용됨
    """
    target_timezone = ZoneInfo(timezone)

    now = datetime.now(target_timezone).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f"{timezone} ({location}) 현재 시각 {now}"

    return location_and_local_time

In [ ]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: TSLA)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max)")

In [9]:
import yfinance as yf

tsla = yf.Ticker("TSLA")

In [16]:
tsla.history(period="5d")

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-01-26 00:00:00-05:00,445.000000,445.040009,434.279999,435.200012,49397400,0.0,0.0
2026-01-27 00:00:00-05:00,437.410004,437.519989,430.690002,430.899994,37733100,0.0,0.0
2026-01-28 00:00:00-05:00,431.910004,438.260010,430.100006,431.459991,54857400,0.0,0.0
2026-01-29 00:00:00-05:00,437.799988,440.230011,414.619995,416.559998,81686100,0.0,0.0
2026-01-30 00:00:00-05:00,425.350006,439.880005,422.700012,430.410004,82483000,0.0,0.0


In [17]:
tsla.history(start="2026-01-01", end="2026-01-10")

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-01-02 00:00:00-05:00,457.799988,458.339996,435.299988,438.070007,85535400,0.0,0.0
2026-01-05 00:00:00-05:00,447.989990,457.549988,444.570007,451.670013,67940800,0.0,0.0
2026-01-06 00:00:00-05:00,446.380005,448.250000,428.779999,432.959991,89093800,0.0,0.0
2026-01-07 00:00:00-05:00,435.899994,438.369995,431.290009,431.410004,59828800,0.0,0.0
2026-01-08 00:00:00-05:00,427.890015,436.890015,424.369995,435.799988,57041100,0.0,0.0
2026-01-09 00:00:00-05:00,435.950012,449.049988,430.390015,445.010010,67331500,0.0,0.0


In [25]:
# 마크다운 변환 위해 설치
# uv add tabulate

result_markdown = tsla.history(start="2026-01-01", end="2026-01-10").to_markdown()

In [27]:
print(result_markdown)

| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |
|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|
| 2026-01-02 00:00:00-05:00 | 457.8  | 458.34 | 435.3  |  438.07 | 8.55354e+07 |           0 |              0 |
| 2026-01-05 00:00:00-05:00 | 447.99 | 457.55 | 444.57 |  451.67 | 6.79408e+07 |           0 |              0 |
| 2026-01-06 00:00:00-05:00 | 446.38 | 448.25 | 428.78 |  432.96 | 8.90938e+07 |           0 |              0 |
| 2026-01-07 00:00:00-05:00 | 435.9  | 438.37 | 431.29 |  431.41 | 5.98288e+07 |           0 |              0 |
| 2026-01-08 00:00:00-05:00 | 427.89 | 436.89 | 424.37 |  435.8  | 5.70411e+07 |           0 |              0 |
| 2026-01-09 00:00:00-05:00 | 435.95 | 449.05 | 430.39 |  445.01 | 6.73315e+07 |           0 |              0 |


In [28]:
type(result_markdown)

str

In [29]:
@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    return history.to_markdown()

In [30]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[get_current_time, get_yf_stock_history],
    system_prompt="너는 사용자의 질문에 답변을 하기 위해 도구를 사용할 수 있다."
)

In [31]:
res = agent.invoke(
    {"messages": [{"role": "user", "content": "부산은 지금 몇시야?"}]}
)

In [33]:
res["messages"][-1].content

'부산의 현재 시각은 2026-01-31 17:27:08 입니다.'

In [34]:
res = agent.invoke(
    {"messages": [{"role": "user", "content": "테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"}]}
)

In [35]:
res

{'messages': [HumanMessage(content='테슬라는 한달 전에 비해 주가가 올랐나 내렸나?', additional_kwargs={}, response_metadata={}, id='1d7a822e-99f3-47ef-8c2d-5cb2543501d6'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_yf_stock_history', 'arguments': '{"stock_history_input": {"period": "1mo", "ticker": "TSLA"}}'}, '__gemini_function_call_thought_signatures__': {'9ebc1de1-4040-4ea3-9095-a09b2fe97ea0': 'CusFAXLI2nwYN0SipBmp7UvT95PKXK5EDi5FFY2qNGTrhHc74dXK98RQ/y0HjFnqWUrJX7thGP/6SboIn2xyTDf3aeuQJGBfKeyOxCXYExavFsKKPnY7rFjubWjpdCSWjx8qt9QghkALYdFAc8x5/FfALE49ejJCfVX2Mlxhc0ZZQkvxqtrI1TSlxestPk6WYqAdmQ8EnpvYoywdUQaLflw4qSiKQb7+6mA1cWSztA0mo5jKS59gkySZYGqxkMSvMtAY3uwkn6iOOTvGz11SaJMFrs23ZyQT1ZNtK66WLqfONMZlmf631/8jVu4kQhFJlUEmg2J0QcfaatooIjyKGZNbZ+YWjpSJT2wiwUUFHzT4YYzcGXOO4XaOmy8SD5CHRE7Ju3XjWd2FXoUaZL53l9OONccoM0SFhT+DHGxpUonzUEt8iXo2pxIoFHGgwvI0qSRsjvusVnTdNOCVdTWs7xo/3NnZKEe/CZQcvKwXRoLOdv8ZYf7CW4vF71j5hJBtuhV6AJCGGm6IwG8ZEVuQqHraZidzGCrXBYkCZA8ohuR+RRFlyibZAGLYBYL7sUeG63W4OQGt2yQ